In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy 
import cansig 
import matplotlib.pyplot as plt
import pathlib as pl
import os

Global seed set to 0


In [2]:
import benchmark.analysis.cnv as cnv
import benchmark.metrics.cnvmetrics as cnvmetrics
import benchmark.metrics.cellmetrics as cellmetrics
import benchmark.metrics.genemetrics as genemetrics
import benchmark.plot.plot_utils as plot_utils
from benchmark.plot.single_plots import *
import benchmark.base.utils as utils


# 1. Setting Parameters & Configuration

In [3]:
# Set these to the file we want to analyze (need to be in folder /data, with the structure /data/DATASET/FILENAME.h5ad)
DATASET = 'simulated_splatter'
FILENAME = 'simulated_splatter.h5ad' 

# inferCNV arguments
STEP = 5
WINDOW_SIZE = 100
REFERENCE_KEY = 'program'
REFERENCE_CAT = ['Plasma', 'Macro']

# Threshold for maximum distance between two cnv segments for merging into connected region
THRESHOLD = 5000000

# 2. Data Preparation 

## 2.1 Read Data

In [4]:
path = !pwd
path = path[0]
datapath = os.path.join(path, 'data', DATASET)
path

'/notebook/Z'

In [5]:
adatas = []


#read external data
externalpath = os.path.join(path, 'data','external')
chr_sizes = pd.read_csv(os.path.join(externalpath, 'chromosome_sizes.csv')).set_index('chr')


#read adata
for num in range(1,21):
    adatas.append(sc.read_h5ad(os.path.join(datapath, 'datasets', 'patient' + str(num) +'.h5ad')))
    

adata = adatas[0].concatenate(*adatas[1:])

/usr/local/lib/python3.8/dist-packages/anndata/_core/anndata.py:1763: FutureWarning: The AnnData.concatenate method is deprecated in favour of the anndata.concat function. Please use anndata.concat instead.

See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  warnings.warn(


## 2.2 Bring data into right format

Each gene in the AnnData object is expected to have a gene location annotation in adata.var in the form of three columns: chromosome, start and end. The chromosome needs to be formatted as "chr<number of the chromosome>". Each cell in the AnnData object has to have some type of cell type annotation to define references group for inferring CNVs, the column of which can be specified in the Settings & Parameters section of the notebook. Additional adata.obs columns needed are 'subclonal', 'malignant_key' and 'Patient'. The count matrix should be stored in compressed sparse row format.

In [6]:
#bring data into right format (ADJUST TO DATA USED)

#dataframe needs: 'subclonal', 'malignant_key', 'Patient'
utils.rename_subclonal(adata, 'subclone', 'batch')
adata.obs.rename(columns={'subclone':'subclonal', 'sample_id': 'Patient'}, inplace = True)

#also expects adata.X to be sparse (Compressed Sparse Row format)
adata.X = scipy.sparse.csr_matrix(adata.X)



# 3. Infer subclonal CNVs


## 3.1 Calculate CNVs

In [7]:
# determine whether gene used to call CNV
adata.var['mean_counts_per_gene'] = np.asarray(adata.X.mean(0)).ravel()
adata.var['cnv_called'] = cnv.get_cnv_called(adata, excluded_chromosomes = ("chrX", "chrY",'chrM'), count_threshold =0.1)

In [8]:
# Make copy, so we can keep original adata 
bdata = adata.copy()

# Normalize bdata, before calling infercnv()
utils.normalize_adata(bdata, 1e4)

# Subset to genes used for inferring cnv
bdata = bdata[:, bdata.var['cnv_called']]

chr_pos, X_cnv = cnv.infercnv(
                bdata,
                reference_key= REFERENCE_KEY,
                reference_cat= REFERENCE_CAT,
                step=STEP,
                window_size=WINDOW_SIZE,
                inplace=False,
                exclude_chromosomes=("chrX", "chrY",'chrM'),
            )

cnv.set_cnv(adata,chr_pos, X_cnv, STEP, WINDOW_SIZE)
cnv.set_subclonal_cnv(adata, cnv_key = 'X_cnv', subclonal_key = 'subclonal')

  0%|          | 0/6 [00:00<?, ?it/s]

## 3.2 Saving CNVs to .h5ad file

In [9]:
# Save adata with inferred CNVs 
interimpath = os.path.join(datapath, 'interim')

isExist = os.path.exists(interimpath)
if not isExist:
   os.makedirs(interimpath)

adata.write_h5ad(os.path.join(interimpath, 'subclonalcnv_step' + str(STEP) + '_' + FILENAME))

... storing 'subclonal' as categorical


## 3.3 Free memory

In [10]:
# To save memory (if adata very big)
del bdata
del chr_pos
del X_cnv
import gc 
gc.collect()

352

# 4. Calculate consecutive CNV regions

## 4.1 Calculate CNV mapping

In [11]:
# Get mapping from inferCNV segments to chromosome positions
cnv_mapping = cnv.get_cnv_mapping(adata)
cnv_mapping['num_genes'] = cnv_mapping.apply(lambda x: cnv.get_num_genes(adata,x), axis = 1)
cnv_mapping = cnv_mapping.transpose()
cnv_mapping.to_csv(os.path.join(interimpath, 'cnv_mapping_step' + str(STEP) +'.csv'))

## 4.2 Calculate initial boundaries of regions

In [12]:
# Get indices of boundaries of possibly connected regions (consists of indices of chromosome boundaries and indices
# between 2 CNV segements whos distance is greater than threshold.

# Calculate these boundaries before even looking at gains/losses, as they are the same for all subclonal groups. 

chr_index = np.sort(np.array(list(adata.uns['cnv']['chr_pos'].values())))
splits = cnvmetrics.get_boundaries(cnv_mapping,chr_index, THRESHOLD)

## 4.3 Calculate regions per subclonal group

In [13]:
# Reduce X_cnv to one row per subclonal group (calculate regions only once per subclonal, later project back) 
subclonal_CNV = pd.DataFrame.sparse.from_spmatrix(adata.obsm['X_cnv']).apply(np.asarray, axis = 0)
subclonal_CNV['subclonal'] = adata.obs.subclonal.values
subclonal_CNV.drop_duplicates(inplace = True)
subclonal_CNV.set_index('subclonal', inplace = True)

gain_regions_unique, loss_regions_unique = cnvmetrics.get_regions(subclonal_CNV, splits, cnv_mapping, THRESHOLD)

loss_regions_unique['type'] = 'loss'
gain_regions_unique['type'] = 'gain'
loss_regions_unique = cnvmetrics.get_region_metrics(adata, cnv_mapping, chr_sizes, loss_regions_unique)
gain_regions_unique = cnvmetrics.get_region_metrics(adata, cnv_mapping, chr_sizes, gain_regions_unique)

## 4.4 Save calculated regions to .csv file


In [14]:
# Save regions 
resultpath = os.path.join(datapath, 'results')

isExist = os.path.exists(resultpath)
if not isExist:
   os.makedirs(resultpath)

regions_unique = pd.concat([gain_regions_unique, loss_regions_unique])
regions_unique.reset_index(inplace = True)
regions_unique.to_csv(os.path.join(resultpath, 'cnv_regions_unique_step' + str(STEP) + '_thresh' + str(THRESHOLD) + '.csv'))

## 4.5 Calculate regions per cell & save


In [15]:
# Project from subclonal to cell-level
subclonal_cellid_mapping = adata.obs[['Patient','subclonal']][adata.obs.subclonal != 'non-malignant'].reset_index(names='cell_id').set_index('subclonal')
loss_regions = subclonal_cellid_mapping.merge(loss_regions_unique,on = 'subclonal', how = 'inner')
gain_regions = subclonal_cellid_mapping.merge(gain_regions_unique,on = 'subclonal', how = 'inner')

regions = pd.concat([gain_regions, loss_regions])
regions.reset_index(inplace = True)
regions.to_csv(os.path.join(resultpath, 'cnv_regions_step' + str(STEP) + '_thresh' + str(THRESHOLD) + '.csv'))


# 5. Calculate CNV metrics per cell & per subclonal group

In [16]:
cnvmetrics_per_cell = cnvmetrics.get_cnvmetrics_per_cell(adata, loss_regions, gain_regions)
cnvmetrics_per_cell.to_csv(os.path.join(resultpath, 'cnvmetrics_per_cell_step' + str(STEP) + '_thresh' + str(THRESHOLD)) + '.csv')

In [17]:
cnvmetrics_per_subclonal = cnvmetrics.get_unique_metrics(cnvmetrics_per_cell)
cnvmetrics_per_subclonal.to_csv(os.path.join(resultpath, 'cnvmetrics_per_subclonal_step' + str(STEP) + '_thresh' + str(THRESHOLD)) + '.csv')

# 6. Calculate normal metrics

## 6.1 Over malignant cells

In [18]:
malignant_adata = adata.copy()
malignant_adata = malignant_adata[utils.get_malignant_index(malignant_adata) ,:]

#metrics on unnormalized data
malignant_adata.raw = malignant_adata
sc.pp.calculate_qc_metrics(malignant_adata, inplace=True)
sc.pp.highly_variable_genes(malignant_adata, n_top_genes = 400, flavor='seurat_v3')
cellmetrics.set_zeroes_genes(malignant_adata)
genemetrics.set_zeroes_cells(malignant_adata)

#metrics on normalized data
utils.normalize_adata(malignant_adata, 1e6)
genemetrics.set_gene_expression_metrics(malignant_adata)
gene_metrics_malignant = malignant_adata.var[['mean_logCPM','var_logCPM','cv','zeroes_cells']]
cell_metrics_malignant = malignant_adata.obs[['malignant_key','zeroes_genes', 'log1p_total_counts']]

#save gene & cell metrics
gene_metrics_malignant.to_csv(os.path.join(resultpath, 'genemetrics_malignant.csv'))
cell_metrics_malignant.to_csv(os.path.join(resultpath, 'cellmetrics_malignant.csv'))

In [19]:
#cell-to-cell correlation 
hv_cell_corr_arr = cellmetrics.get_cell_cell_correlation_hv(malignant_adata)
all_cell_corr_arr = cellmetrics.get_cell_cell_correlation_all(malignant_adata)

cell_corr = pd.DataFrame([hv_cell_corr_arr,all_cell_corr_arr], index = ['hv','all'])
cell_corr.transpose().to_csv(os.path.join(resultpath, 'cell_corr_malignant.csv'))

In [20]:
#gene-to-gene correlation
he_gene_corr_arr = genemetrics.get_gene_gene_correlation_he(malignant_adata)
hv_gene_corr_arr = genemetrics.get_gene_gene_correlation_hv(malignant_adata)

gene_corr = pd.DataFrame([he_gene_corr_arr,hv_gene_corr_arr], index = ['he','hv'])
gene_corr.transpose().to_csv(os.path.join(resultpath ,'gene_corr_malignant.csv'))

In [21]:
#free memory 
del malignant_adata
gc.collect()

465

## 6.2 Over all cells 


In [22]:
#metrics on unnormalized data
adata.raw = adata
sc.pp.calculate_qc_metrics(adata, inplace=True)
sc.pp.highly_variable_genes(adata, n_top_genes = 400, flavor='seurat_v3')
cellmetrics.set_zeroes_genes(adata)
genemetrics.set_zeroes_cells(adata)

#metrics on normalized data
utils.normalize_adata(adata, 1e6)
genemetrics.set_gene_expression_metrics(adata)
gene_metrics_all = adata.var[['mean_logCPM','var_logCPM','cv','zeroes_cells']]
cell_metrics_all = adata.obs[['malignant_key','zeroes_genes', 'log1p_total_counts']]

In [23]:
#save gene & cell metrics
gene_metrics_all.to_csv(os.path.join(resultpath, 'genemetrics_all.csv'))
cell_metrics_all.to_csv(os.path.join(resultpath, 'cellmetrics_all.csv'))

In [24]:
#cell-to-cell correlation 
hv_cell_corr_arr = cellmetrics.get_cell_cell_correlation_hv(adata)
all_cell_corr_arr = cellmetrics.get_cell_cell_correlation_all(adata)

cell_corr = pd.DataFrame([hv_cell_corr_arr,all_cell_corr_arr], index = ['hv','all'])
cell_corr.transpose().to_csv(os.path.join(resultpath, 'cell_corr_all.csv'))

In [25]:
#gene-to-gene correlation
he_gene_corr_arr = genemetrics.get_gene_gene_correlation_he(adata)
hv_gene_corr_arr = genemetrics.get_gene_gene_correlation_hv(adata)

gene_corr = pd.DataFrame([he_gene_corr_arr,hv_gene_corr_arr], index = ['he','hv'])
gene_corr.transpose().to_csv(os.path.join(resultpath, 'gene_corr_all.csv'))

# 7. Plots

In [26]:
plotpath = os.path.join(path, 'plots', DATASET)
resultpath = os.path.join(datapath, 'results')

isExist = os.path.exists(plotpath)
if not isExist:
   os.makedirs(plotpath)

paramstring = '_step' + str(STEP) + '_thresh' + str(THRESHOLD)

In [27]:
df = plot_utils.read_data(os.path.join(resultpath, 'cnv_regions' + paramstring + '.csv'))
plot_cnv_bp(df, percentile=95, output_dir=plotpath, output_name='cnv_regions_bp' + paramstring )
plot_cnv_gene(df, percentile=99.50, output_dir=plotpath, output_name='cnv_regions_gene' + paramstring)


df = read_data(os.path.join(resultpath, 'cnvmetrics_per_subclonal' + paramstring + '.csv'))
plot_stat_bp(df, percentile=75.0, output_dir= plotpath, output_name='cnv_stats' + paramstring)
plot_nregions_per_cell(df, percentile=100.0, output_dir=plotpath, output_name='nregions_per_cell' + paramstring)
plot_cnv_per_cell(df, percentile=100.0, output_dir=plotpath, output_name='cnv_per_cell' + paramstring)

df = read_data(os.path.join(resultpath, 'genemetrics_all.csv'))
plot_expression_stats(df, percentile=99.0, output_dir=plotpath, output_name='genemetrics_all')

df = read_data(os.path.join(resultpath, 'genemetrics_malignant.csv'))
plot_expression_stats(df, percentile=99.0, output_dir=plotpath, output_name='genemetrics_malignant')
                      
df = read_data(os.path.join(resultpath, 'gene_corr_all.csv'))
plot_gene2gene_corr(df, output_dir=plotpath, output_name='gene_corr_all')

df = read_data(os.path.join(resultpath, 'gene_corr_malignant.csv'))
plot_gene2gene_corr(df, output_dir=plotpath, output_name='gene_corr_malignant')


df = read_data(os.path.join(resultpath, 'cellmetrics_all.csv'))
plot_cell_counts(df, output_dir=plotpath, output_name='cellmetrics_all')

df = read_data(os.path.join(resultpath, 'cellmetrics_malignant.csv'))
plot_cell_counts(df, output_dir=plotpath, output_name='cellmetrics_malignant')

df = read_data(os.path.join(resultpath, 'cell_corr_all.csv'))
plot_cell2cell_corr(df, output_dir=plotpath, output_name='cell_corr_all')


df = read_data(os.path.join(resultpath, 'cell_corr_malignant.csv'))
plot_cell2cell_corr(df, output_dir=plotpath, output_name='cell_corr_malignant')




df = read_data(os.path.join(resultpath, 'cnv_regions_unique' + paramstring + '.csv'))
plot_chr_locations(df, output_dir=plotpath, output_name='chr_locations' + paramstring)
plot_chr_location_and_size_all(df, output_dir=plotpath, output_name= 'chr_locations_and_size_all' + paramstring)
plot_chr_location_and_size_gl(df, output_dir=plotpath, output_name= 'chr_locations_and_size_gl' + paramstring)


plt.close()

/notebook/Z/benchmark/plot/single_plots.py:447: RuntimeWarning: invalid value encountered in divide
  bins_chr_n[i, :] /= float(sum_)
/usr/local/lib/python3.8/dist-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/usr/local/lib/python3.8/dist-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
